# MTUQ Data Fetching and Processing Pipeline

This notebook handles the automated acquisition and MTUQ-preparation of seismic data for an event in Louisian on March 5, 2026.

**Workflow:**
1. Load centralized parameters from `project_config.json`.

2. Fetch FDSN continuous data and StationXML files.

3. Remove instrument response, rotate to RTZ, inject MTUQ SAC headers, and build `weights.dat`.

4. Plot synthetic/observed phase arrivals for quality control.

5. Filter weights.dat file based on a distance criteria.

6. Plot record section.

7. Check amplitude decay.

8. Plot map. 


## 1. Load json file and download the data

In [ ]:
from fetch_data_mtuq.core_config import load_config
from fetch_data_mtuq.download import run_download

# 1. Load the master configuration
config = load_config("20260305113008.json")

# 2. Execute the download protocol
print("Initiating FDSN download...")
run_download(config)

## 2. Preprocess data

2.1 Remove instrument response.

2.2 Rotate to radial and transverse.

2.3 Write weight.dat file.

In [ ]:
from fetch_data_mtuq.process import run_processing

# 3. Process the raw MiniSEED data into MTUQ-ready SAC format
print("Initiating MTUQ processing pipeline...")
run_processing(config)

## 3. Plot waveforms for individual stations


In [ ]:
from fetch_data_mtuq.plot import plot_station_data

# 4. Plot waveforms for a single station
# Update 'station_name' with a valid station ID that was successfully processed in the previous step.

station_name = "AG.Z41A.00."  # Replace with an actual station from your processing summary
event_time = config["event"]["time"]
data_directory = "20260305T113008_data_SAC_MTUQ"

# Bandpass limits (Hz)
f_min = 0.005 
f_max = 20.0

# Window relative to trace onset (seconds)
window_t1 = 0
window_t2 = 600 

plot_station_data(
    station_id=station_name, 
    data_dir=data_directory, 
    origin_time_str=event_time, 
    freqmin=f_min, 
    freqmax=f_max, 
    t1=window_t1, 
    t2=window_t2, 
    phases=["p", "s","P","S"],
    output_file=f"QC_plot_{station_name}.pdf"
)


## 4. Filter weights.dat file based on distance criteria

In [ ]:
from fetch_data_mtuq.fetch_mtuq_utils import filter_weight_file

# 1. Define the paths based on your processed event
data_directory = "20260305T113008_data_SAC_MTUQ"
master_weight_file = f"{data_directory}/weights.dat"

# 2. Define the distance bands you want to test (min_km, max_km)
# You can add as many ranges to this list as you need
distance_bands = [
    (0, 100),   # Very near-field
    (100, 300), # Regional
    (0, 500)    # Full dataset
]

# 3. Execute the filter
print(f"Reading master weights from: {master_weight_file}")
filter_weight_file(
    input_weight_file=master_weight_file,
    output_directory=data_directory,
    distance_ranges=distance_bands
)

## 5. Plot record section

In [ ]:
from fetch_data_mtuq.plot import plot_record_section

weight_file="20260305T113008_data_SAC_MTUQ/weights_0_500.dat"
data_dir="20260305T113008_data_SAC_MTUQ" 
f_min = 0.005 
f_max = 20.0 
t1=0
t2=500 
#f_min = 1/50
#f_max = 1/20
output_file="Record_Section_0_500km.pdf"
component='Z'
amp_scale=5.0  # Increase this if the wiggles are too small

# Make sure you point this to one of the weight files you generated earlier
plot_record_section(weight_file, data_dir, f_min,f_max,t1, t2, output_file,component,amp_scale)

## 6. Check amplitude decay

In [ ]:
from fetch_data_mtuq.plot import check_amplitude_decay

weight_file="20260305T113008_data_SAC_MTUQ/weights_0_500.dat"
data_dir="20260305T113008_data_SAC_MTUQ" 
f_min = 0.005 
f_max = 20.0 
t1=0
t2=500 
#f_min = 1/50
#f_max = 1/20
component='Z'

output_name = "QC_Amplitude_Decay_0_500km.pdf"

scale = 'linear'  # Use 'linear' for linear scale or 'log' for logarithmic scale

check_amplitude_decay(weight_file, data_dir, f_min, f_max, t1, t2, output_name, component, scale)

## 7. Plot Stations map

In [ ]:
from fetch_data_mtuq.plot import plot_station_map

weight_file="20260305T113008_data_SAC_MTUQ/weights_0_500.dat"
data_dir="20260305T113008_data_SAC_MTUQ"
map_output_name = "Station_Map_0_100km.pdf"
projection="M15c"  # Mercator projection
component='Z'#This component is used to determine the station locations from the SAC headers. It does not affect the map itself, which only plots station locations and the event epicenter.

# Generate the map
# Generate the map using explicit keyword arguments
plot_station_map(weight_file, data_dir, map_output_name,component,projection)